# LLM Model Benchmarking Pipeline

In [ ]:
MODEL_NAME = "gemma3:270m"
input_path = "/workspaces/Project/CSDS555-ResAI-Final-Project-Research/data/input/test_data.json"

## Ollama Setup

In [2]:
%%bash
nvidia-smi

Tue Nov  4 22:01:39 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...    Off |   00000000:01:00.0  On |                  N/A |
| N/A   38C    P5              7W /  100W |     526MiB /   6141MiB |     26%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# %%bash
# curl -fsSL https://ollama.com/install.sh | sh

In [4]:
import requests
import time
import subprocess

In [5]:
# Start Ollama server in background
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Wait until server responds
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=1)
        if r.ok:
            print("[ollama server] Ollama server ready")
            break
    except Exception as _:
        print(f"[ollama server] Waiting for server... ({i+1}/30)")
        time.sleep(1)

[ollama server] Waiting for server... (1/30)
[ollama server] Ollama server ready


In [6]:
# List of models you want to download
print(f"[model download] Downloading {MODEL_NAME} ...")
try:
    # Run the Ollama pull command
    result = subprocess.run(
        ["ollama", "pull", MODEL_NAME],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=True
    )
    print(f"[model download] Successfully downloaded {MODEL_NAME}\n")
except subprocess.CalledProcessError as e:
    print(f"[model download] Failed to download {MODEL_NAME}")
    print(e.stderr)


[model download] Downloading gemma3:270m ...
[model download] Successfully downloaded gemma3:270m



In [7]:
# Test URL
url = "http://localhost:11434/api/tags"

print("Checking if Ollama server is running...")

for i in range(30):  # wait up to ~30 seconds
    try:
        r = requests.get(url, timeout=2)
        if r.ok:
            print("[ollama server] Ollama server is running!")
            print("Available models:")
            k = r.json()["models"]
            for a, m in enumerate(k):
                print(a, m["name"])
            break
    except Exception as _:
        print(f"[ollama server] Waiting for server... ({i+1}/30)")
        time.sleep(1)
else:
    print("[ollama server] Ollama server not responding.")


Checking if Ollama server is running...
[ollama server] Ollama server is running!
Available models:
0 gemma3:270m
1 mistral:7b
2 gemma3:12b
3 llama3.1:8b
4 qwen3:8b


In [8]:
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(model=MODEL_NAME, messages=[
  {
    'role': 'user',
    'content': 'Why is the sky blue?',
  },
])
print(response['message']['content'])
# or access fields directly from the response object
print(response.message.content)

The sky is blue because of a phenomenon called **Rayleigh scattering**. 

Here's a breakdown:

*   **Sunlight:** Sunlight is made up of all the colors of the rainbow.
*   **Rayleigh Scattering:** When sunlight hits the Earth's atmosphere, it collides with tiny particles of dust, gases, and other particles suspended in the air. These particles scatter the light in all directions, creating a rainbow-like effect.
*   **Blue Light:** The blue and violet colors of sunlight are scattered much more effectively than the red and orange colors. This is because the blue and violet light have shorter wavelengths.

Therefore, the sky appears blue because the sunlight that reaches our eyes is scattered in all directions by the air molecules.
The sky is blue because of a phenomenon called **Rayleigh scattering**. 

Here's a breakdown:

*   **Sunlight:** Sunlight is made up of all the colors of the rainbow.
*   **Rayleigh Scattering:** When sunlight hits the Earth's atmosphere, it collides with tiny p

## Importing Packages for Execution

In [9]:
import os
import json
import time

from pathlib import Path
from ollama import generate

## Utility Functions

In [10]:
def json_to_dict(file_path: str) -> dict:
    data_dict = {}
    with open(file_path, 'r') as data:
        data_dict = json.load(data)

    return data_dict

In [11]:
def csv_to_dict(file_path: str) -> dict:
    data_list = []
    data_dict = {}

    with open(file_path, 'r') as data:
        data_list = data.readlines()

    for elements in data_list[1:]:
        id, sys, prompt = elements.split(',')
        data_dict[id] = {
            "system_prompt": sys.strip(),
            "test_prompt": prompt.strip()
        }

    return data_dict

In [12]:
def inference_llm(sys_prompt: str, text_prompt: str, model_name=MODEL_NAME) -> str:
    output = generate(model=model_name, prompt=text_prompt, system=sys_prompt)
    return output.response
    

## Training Setup
- Goal: read different prompts from csv file and do inferences, save the output to another csv file with unique_id.
- This is a skeleton file, a singular function should trigger the training process and save the output to csv at every k prompts.

### Steps:
1. Load csv data into dictionary {unique_id: {system_prompt: "prompt", test_prompt: "prompt"}}
2. Loop through data and call llm_inference function to get output from llm
3. Store output into dictionary {unique_id: output}
4. Save the dictionary at every k prompts in a csv

In [13]:
def benchmark_data(input_file: str, model_name: str=MODEL_NAME) -> str:
    """
    Benchmarks the given dataset on given model
    
    Args:
        input_file: Absolute path to input data to benchmark
        model_name: Name of Ollama Model to benchmark
    
    Returns:
        str: path where output is stored
    """
    path = Path(input_file)  
    filename, filetype = path.name.split('.')  # abc.xyz
    data_dir = path.parent.parent  # Parent of input dir [CSDS555/data/]

    out_name = f"{filename}_{time.time_ns()}.json"  # output filename same as input with timestamp
    output_dir = os.path.join(data_dir, "output")  # Output dir [CSDS555/data/output/]
    output_file = os.path.join(output_dir, out_name)

    # Create output dir if it doesn't exist
    if not os.path.exists(output_dir):
        print(f"Creating output directory: {output_dir}")
        os.makedirs(output_dir)
    
    # Step 1: Load data
    if filetype == "csv":
        input_dict = csv_to_dict(input_file)
    elif filetype == "json":
        input_dict = json_to_dict(input_file)
    else:
        raise TypeError(f"Expected filetype 'csv' or 'json', got {filetype}")

    # Step 2: Inferencing data in loop
    output_data = {}
    for i, (id, prompt) in enumerate(input_dict.items()):
        if (i % 5) == 0:
            with open(output_file, 'w') as op_file:
                json.dump(output_data, op_file, indent=2)
        
        output = inference_llm(sys_prompt=prompt["system_prompt"], text_prompt=prompt["test_prompt"])
        output_data[id] = output

    # Final load if any are missed
    with open(output_file, 'w') as op_file:
        json.dump(output_data, op_file, indent=2)

    return output_file

## Execution Cell

In [ ]:
benchmark_data(input_file=input_path)

'/workspaces/Project/CSDS555-ResAI-Final-Project-Research/data/output/test_data_1762293738182279398.json'